In [1]:
import pandas as pd
import numpy as np
import sklearn
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Pre-processing

In [3]:
df = pd.read_csv("Suicide_Detection (1).csv")

# Rename 'Unnamed' column
df = df.rename(columns={'Unnamed: 0': 'ID'})

# Drop rows with missing values in the 'class' column
df = df.dropna(subset=['class'])

In [4]:
df.head()

,ID,text,class
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide
1,3,Am I weird I don't get affected by compliments...,non-suicide
2,4,Finally 2020 is almost over... So I can never ...,non-suicide
3,8,i need helpjust help me im crying so hard,suicide
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide


In [5]:
df.isnull().sum()

ID       0
text     0
class    0
dtype: int64

In [6]:
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove non-alphabetic characters
    text = ' '.join(text.split())  # Remove extra whitespaces
    return text

# Apply the cleaning function to the 'text' column
df['cleaned_text'] = df['text'].apply(clean_text)

# Tokenization
df['tokenized_text'] = df['cleaned_text'].apply(word_tokenize)

# Lowercasing
df['lowercased_text'] = df['tokenized_text'].apply(lambda x: [word.lower() for word in x])

# Removing stopwords
stop_words = set(stopwords.words('english'))
df['filtered_text'] = df['lowercased_text'].apply(lambda x: [word for word in x if word not in stop_words])

# Lemmatization
lemmatizer = WordNetLemmatizer()
df['lemmatized_text'] = df['filtered_text'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

In [7]:
df.head()

,ID,text,class,cleaned_text,tokenized_text,lowercased_text,filtered_text,lemmatized_text
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide,Ex Wife Threatening SuicideRecently I left my ...,"[Ex, Wife, Threatening, SuicideRecently, I, le...","[ex, wife, threatening, suiciderecently, i, le...","[ex, wife, threatening, suiciderecently, left,...","[ex, wife, threatening, suiciderecently, left,..."
1,3,Am I weird I don't get affected by compliments...,non-suicide,Am I weird I dont get affected by compliments ...,"[Am, I, weird, I, dont, get, affected, by, com...","[am, i, weird, i, dont, get, affected, by, com...","[weird, dont, get, affected, compliments, comi...","[weird, dont, get, affected, compliment, comin..."
2,4,Finally 2020 is almost over... So I can never ...,non-suicide,Finally is almost over So I can never hear has...,"[Finally, is, almost, over, So, I, can, never,...","[finally, is, almost, over, so, i, can, never,...","[finally, almost, never, hear, bad, year, ever...","[finally, almost, never, hear, bad, year, ever..."
3,8,i need helpjust help me im crying so hard,suicide,i need helpjust help me im crying so hard,"[i, need, helpjust, help, me, im, crying, so, ...","[i, need, helpjust, help, me, im, crying, so, ...","[need, helpjust, help, im, crying, hard]","[need, helpjust, help, im, cry, hard]"
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide,Im so lostHello my name is Adam and Ive been s...,"[Im, so, lostHello, my, name, is, Adam, and, I...","[im, so, losthello, my, name, is, adam, and, i...","[im, losthello, name, adam, ive, struggling, y...","[im, losthello, name, adam, ive, struggling, y..."


In [8]:
#Unique values in class
print(df['class'].unique())
# Create binary for class column
df['class_binary'] = df['class'].map({'suicide': 1, 'non-suicide': 0})

['suicide' 'non-suicide']


In [9]:
df.head()

,ID,text,class,cleaned_text,tokenized_text,lowercased_text,filtered_text,lemmatized_text,class_binary
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide,Ex Wife Threatening SuicideRecently I left my ...,"[Ex, Wife, Threatening, SuicideRecently, I, le...","[ex, wife, threatening, suiciderecently, i, le...","[ex, wife, threatening, suiciderecently, left,...","[ex, wife, threatening, suiciderecently, left,...",1
1,3,Am I weird I don't get affected by compliments...,non-suicide,Am I weird I dont get affected by compliments ...,"[Am, I, weird, I, dont, get, affected, by, com...","[am, i, weird, i, dont, get, affected, by, com...","[weird, dont, get, affected, compliments, comi...","[weird, dont, get, affected, compliment, comin...",0
2,4,Finally 2020 is almost over... So I can never ...,non-suicide,Finally is almost over So I can never hear has...,"[Finally, is, almost, over, So, I, can, never,...","[finally, is, almost, over, so, i, can, never,...","[finally, almost, never, hear, bad, year, ever...","[finally, almost, never, hear, bad, year, ever...",0
3,8,i need helpjust help me im crying so hard,suicide,i need helpjust help me im crying so hard,"[i, need, helpjust, help, me, im, crying, so, ...","[i, need, helpjust, help, me, im, crying, so, ...","[need, helpjust, help, im, crying, hard]","[need, helpjust, help, im, cry, hard]",1
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide,Im so lostHello my name is Adam and Ive been s...,"[Im, so, lostHello, my, name, is, Adam, and, I...","[im, so, losthello, my, name, is, adam, and, i...","[im, losthello, name, adam, ive, struggling, y...","[im, losthello, name, adam, ive, struggling, y...",1


In [10]:
df.isnull().sum()

ID                 0
text               0
class              0
cleaned_text       0
tokenized_text     0
lowercased_text    0
filtered_text      0
lemmatized_text    0
class_binary       0
dtype: int64

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine the lemmatized words back into sentences
df['processed_text'] = df['lemmatized_text'].apply(lambda x: ' '.join(x))

# Initialize the TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)  # You can adjust max_features as needed

# Fit and transform the processed text
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])

# Convert the TF-IDF matrix to a DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# Concatenate the TF-IDF DataFrame with the original DataFrame
df = pd.concat([df, tfidf_df], axis=1)

In [12]:
df.head()

,ID,text,class,cleaned_text,tokenized_text,lowercased_text,filtered_text,lemmatized_text,class_binary,processed_text,...,youtuber,youve,yr,yt,yum,zero,zoloft,zombie,zone,zoom
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide,Ex Wife Threatening SuicideRecently I left my ...,"[Ex, Wife, Threatening, SuicideRecently, I, le...","[ex, wife, threatening, suiciderecently, i, le...","[ex, wife, threatening, suiciderecently, left,...","[ex, wife, threatening, suiciderecently, left,...",1,ex wife threatening suiciderecently left wife ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3,Am I weird I don't get affected by compliments...,non-suicide,Am I weird I dont get affected by compliments ...,"[Am, I, weird, I, dont, get, affected, by, com...","[am, i, weird, i, dont, get, affected, by, com...","[weird, dont, get, affected, compliments, comi...","[weird, dont, get, affected, compliment, comin...",0,weird dont get affected compliment coming some...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4,Finally 2020 is almost over... So I can never ...,non-suicide,Finally is almost over So I can never hear has...,"[Finally, is, almost, over, So, I, can, never,...","[finally, is, almost, over, so, i, can, never,...","[finally, almost, never, hear, bad, year, ever...","[finally, almost, never, hear, bad, year, ever...",0,finally almost never hear bad year ever swear ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,8,i need helpjust help me im crying so hard,suicide,i need helpjust help me im crying so hard,"[i, need, helpjust, help, me, im, crying, so, ...","[i, need, helpjust, help, me, im, crying, so, ...","[need, helpjust, help, im, crying, hard]","[need, helpjust, help, im, cry, hard]",1,need helpjust help im cry hard,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide,Im so lostHello my name is Adam and Ive been s...,"[Im, so, lostHello, my, name, is, Adam, and, I...","[im, so, losthello, my, name, is, adam, and, i...","[im, losthello, name, adam, ive, struggling, y...","[im, losthello, name, adam, ive, struggling, y...",1,im losthello name adam ive struggling year im ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Assuming 'class_binary' is the target variable
X = df.drop(['ID', 'text', 'class', 'class_binary', 'cleaned_text', 'tokenized_text', 'lowercased_text', 'filtered_text', 'lemmatized_text', 'processed_text'], axis=1)
y = df['class_binary']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train a logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions on the testing set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Confusion Matrix:\n{conf_matrix}")
print(f"Classification Report:\n{classification_rep}")

Accuracy: 0.901595744680851
Confusion Matrix:
[[701  48]
 [100 655]]
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.94      0.90       749
           1       0.93      0.87      0.90       755

    accuracy                           0.90      1504
   macro avg       0.90      0.90      0.90      1504
weighted avg       0.90      0.90      0.90      1504



# Applying Basic Machine Learning Algorithms:

# 1. Decision Tree

In [15]:
from sklearn.tree import DecisionTreeClassifier

X = df.drop(['ID', 'text', 'class', 'class_binary', 'cleaned_text', 'tokenized_text', 'lowercased_text', 'filtered_text', 'lemmatized_text', 'processed_text'], axis=1)
y = df['class_binary']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a decision tree classifier
clf = DecisionTreeClassifier()

# Train the classifier on the training data
clf.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = clf.predict(X_test)

# Evaluate the accuracy of the classifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test,y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')
print('Confusion Matrix:')
print(conf_matrix)


Accuracy: 0.80
Precision: 0.80
Recall: 0.80
F1 Score: 0.80
Confusion Matrix:
[[597 152]
 [153 602]]


# 2. Random Forest

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
# Assuming 'class_binary' is the target variable
X = df.drop(['ID', 'text', 'class', 'class_binary', 'cleaned_text', 'tokenized_text', 'lowercased_text', 'filtered_text', 'lemmatized_text', 'processed_text'], axis=1)
y = df['class_binary']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the classifier on the training data
clf.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = clf.predict(X_test)

# Evaluate the accuracy of the classifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test,y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')
print('Confusion Matrix:')
print(conf_matrix)

Accuracy: 0.88
Precision: 0.89
Recall: 0.88
F1 Score: 0.88
Confusion Matrix:
[[667  82]
 [ 94 661]]


# 3. K-Neighbours Classifier

In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
# Assuming 'class_binary' is the target variable
X = df.drop(['ID', 'text', 'class', 'class_binary', 'cleaned_text', 'tokenized_text', 'lowercased_text', 'filtered_text', 'lemmatized_text', 'processed_text'], axis=1)
y = df['class_binary']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a KNN classifier (you can adjust the 'n_neighbors' parameter)
knn = KNeighborsClassifier(n_neighbors=5)

# Train the classifier on the training data
knn.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = knn.predict(X_test)

# Evaluate the accuracy of the classifier
accuracy = accuracy_score(y_test, y_pred)
confusion_mat = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Confusion Matrix:\n", confusion_mat)


Accuracy: 0.5006648936170213
Confusion Matrix:
 [[745   4]
 [747   8]]


# Finally, an Ensemble model is created that uses voting classifier to combine all the previous three models to give the most accurate output

In [17]:
from sklearn.ensemble import VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Assuming 'class_binary' is the target variable
X = df.drop(['ID', 'text', 'class', 'class_binary', 'cleaned_text', 'tokenized_text', 'lowercased_text', 'filtered_text', 'lemmatized_text', 'processed_text'], axis=1)
y = df['class_binary']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize individual models
decision_tree = DecisionTreeClassifier(random_state=42)
random_forest = RandomForestClassifier(random_state=42)
knn = KNeighborsClassifier()

# Create an ensemble model using a soft voting strategy
ensemble_model = VotingClassifier(
    estimators=[('decision_tree', decision_tree), ('random_forest', random_forest), ('knn', knn)],
    voting='soft'  # Soft voting takes the average of probabilities
)

# Train the ensemble model
ensemble_model.fit(X_train, y_train)

# Make predictions on the testing set
y_pred_ensemble = ensemble_model.predict(X_test)

# Evaluate the ensemble model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred_ensemble)
precision = precision_score(y_test, y_pred_ensemble)
recall = recall_score(y_test, y_pred_ensemble)
f1 = f1_score(y_test,y_pred_ensemble)
conf_matrix = confusion_matrix(y_test, y_pred_ensemble)

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')
print('Confusion Matrix:')
print(conf_matrix)


Accuracy: 0.80
Precision: 0.81
Recall: 0.79
F1 Score: 0.80
Confusion Matrix:
[[612 137]
 [157 598]]
